# 🧱 Structured Output — typed JSON from the model

The model normally replies with prose, but applications need **typed data**. The
reliable pattern in `pi-ai` is to define **one tool whose parameters _are_ your
output schema**, instruct the model to answer only by calling it, then run
`validateToolCall()` to get a validated, strongly-typed object back.

This is far more robust than asking for JSON in text and `JSON.parse`-ing it:
the schema is enforced, and a bad shape throws an error you can catch and retry.

> **Kernel:** Deno

## 1. Load env & register Azure OpenAI

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

In [ ]:
import { registerAzure } from "playground/azure";
const { models, model, modelId } = registerAzure();

## 2. A tool whose parameters are the output schema

We describe the data we want with **TypeBox** (`Type.*`), wrap it as a `Tool`,
and derive a compile-time TypeScript type from the very same schema with
`Static<...>` — one source of truth for both the runtime validation and the
static types.

In [ ]:
import { Type, type Static, type Tool, type Context, validateToolCall } from "@earendil-works/pi-ai";

// The tool's *parameters* ARE our desired output shape. We will tell the model
// to answer only by calling this tool, so its "reply" becomes a typed object.
const ExtractPerson = {
  name: "extract_person",
  description: "Record the structured details of the person described.",
  parameters: Type.Object({
    fullName: Type.String(),
    age: Type.Optional(Type.Number()),
    occupation: Type.String(),
    skills: Type.Array(Type.String()),
    city: Type.Optional(Type.String()),
  }),
} satisfies Tool;

// Derive the compile-time TypeScript type straight from the schema.
type Person = Static<typeof ExtractPerson.parameters>;

console.log("Schema ready ✔  fields:", Object.keys(ExtractPerson.parameters.properties).join(", "));

## 3. Force the tool call and validate the result

The `systemPrompt` instructs the model to respond **only** by calling
`extract_person`. We pass a messy, unstructured bio as the user message. After
`completeSimple`, we locate the `toolCall` block and hand it to
`validateToolCall([tool], call)` — which checks the arguments against the schema
and returns them. We cast to `Person` so the rest of the code is fully typed.

In [ ]:
const bio =
  "So this is Gianna Rossi — she's 34, has been writing Rust and TypeScript for " +
  "years and now leads a small platform team. Lives up in Turin. Off the clock " +
  "she's into freediving and woodworking.";

const personContext: Context = {
  systemPrompt:
    "You are a data-extraction engine. ALWAYS respond by calling the " +
    "`extract_person` tool with the details you find. Do not write any prose.",
  messages: [{ role: "user", content: bio, timestamp: Date.now() }],
  tools: [ExtractPerson],
};

const personReply = await models.completeSimple(model, personContext);
console.log("stopReason:", personReply.stopReason);

const personCall = personReply.content.find((b) => b.type === "toolCall");
if (!personCall || personCall.type !== "toolCall") {
  throw new Error(
    "Model did not call extract_person. Raw content: " + JSON.stringify(personReply.content),
  );
}

// Validate raw arguments against the TypeBox schema -> typed object (throws on mismatch).
const person = validateToolCall([ExtractPerson], personCall) as Person;

console.log("\n👤 Extracted person (typed):");
console.log(person);
console.log(`\nfullName : ${person.fullName}`);
console.log(`skills (${person.skills.length}): ${person.skills.join(", ")}`);
console.log(
  `\ntokens: ${personReply.usage.input} in / ${personReply.usage.output} out` +
    `  |  cost: $${personReply.usage.cost.total.toFixed(6)}`,
);

## 4. Same pattern, different schema — ticket triage

The pattern is fully reusable: swap in another schema and prompt. Here we
classify a support ticket into fixed enums using `StringEnum`, so `category`
and `urgency` are constrained to known string literals (and typed as unions).

In [ ]:
import { StringEnum } from "@earendil-works/pi-ai";

const ClassifyTicket = {
  name: "classify_ticket",
  description: "Classify an incoming support ticket.",
  parameters: Type.Object({
    category: StringEnum(["billing", "bug", "feature", "other"]),
    urgency: StringEnum(["low", "medium", "high"]),
    summary: Type.String({ description: "One-sentence summary of the issue." }),
  }),
} satisfies Tool;

type Ticket = Static<typeof ClassifyTicket.parameters>;

const ticketText =
  "URGENT: after the latest update the export button throws a 500 and I " +
  "can't get my invoices out before the end-of-month deadline. Please help!";

const ticketContext: Context = {
  systemPrompt:
    "You triage support tickets. ALWAYS respond by calling the " +
    "`classify_ticket` tool. Do not write prose.",
  messages: [{ role: "user", content: ticketText, timestamp: Date.now() }],
  tools: [ClassifyTicket],
};

const ticketReply = await models.completeSimple(model, ticketContext);
const ticketCall = ticketReply.content.find((b) => b.type === "toolCall");
if (!ticketCall || ticketCall.type !== "toolCall") {
  throw new Error("Model did not call classify_ticket.");
}

const ticket = validateToolCall([ClassifyTicket], ticketCall) as Ticket;
console.log("🎫 Classified ticket (typed):");
console.log(ticket);
console.log(`\n→ ${ticket.category} / ${ticket.urgency}: ${ticket.summary}`);

## ✅ What just happened

1. We defined a **`Tool` whose `parameters` are the output schema** (TypeBox), and
   derived a matching TypeScript type with `Static<...>`.
2. A `systemPrompt` forced the model to reply **only** by calling that tool.
3. `validateToolCall([tool], call)` validated the model's arguments against the
   schema and returned them — we cast to our type for full editor support.
4. The same three steps worked unchanged for a second schema (ticket triage with
   `StringEnum`), showing the pattern is reusable for any structured task.

Because validation is explicit, malformed output throws where you can **catch it
and retry**, instead of silently feeding bad data downstream.

Next: **060 — Vision / image input**.